In [ ]:
import numpy as np
import pandas as pd
import math
import heapq
import matplotlib.pyplot as plt
from decimal import Decimal
floaterr = 1.0e-8

In [ ]:
# a = pd.read_csv("./datasets/a280.csv")
# a = pd.read_csv("./minidatasets/minitsp.csv")
# a = pd.read_csv("./datasets/xql662.csv")
a = pd.read_csv("./datasets/kz9976.csv")
points = np.array(a[['x','y']])
inputsize = len(points)

In [ ]:
# distance matrix 만들기
diff = points[:, np.newaxis, :] - points[np.newaxis, :, :]
distance_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))

In [ ]:
# MST 찾기( 프림 알고리즘 사용)
visited = [False] * len(points)
minheap = []
mstgraph = np.full( ( inputsize, inputsize ), np.inf)

# 0 추가
for x in range(inputsize):
    heapq.heappush(minheap, ( distance_matrix[0][x] , 0, x ) )
visited[ 0 ] = True

while minheap:
    Weight, From, To = heapq.heappop( minheap )
    if not visited[To]:
        visited[To] = True
        mstgraph[ From, To ] = Weight
        mstgraph[ To, From ] = Weight
        for x in range(inputsize):
            next_to, next_weight = x, distance_matrix[To][x]
            if not visited[ next_to ]:
                heapq.heappush( minheap, ( next_weight, To, next_to ) )

In [ ]:
finite_mask = np.isfinite(mstgraph)
total_weight = np.sum(mstgraph[finite_mask]) / 2

print("MST 총 가중치:", total_weight)

In [ ]:
# 홀수 degree 찾기
oddmat = np.full( ( inputsize, inputsize ), np.inf)

finite_mask = np.isfinite(mstgraph)              # inf가 아닌 값은 True
finite_count_per_row = finite_mask.sum(axis=1) # 각 행마다 inf가 아닌 값의 개수
odd_rows = np.where(finite_count_per_row % 2 == 1)[0]  # 개수가 홀수인 행 인덱스

print(odd_rows.tolist())

In [ ]:
next_id = len(distance_matrix)

In [ ]:
class blossom:
    def __init__(self, G, Id, nodes=None):
        self.Graph = G
        self.Id = Id
        self.Dual = Decimal(0)
        self.Matching = None
        self.Nodes = [] # 블라썸에 포함된 노드들
        
        self.Visited = False
        self.Label = True
        self.parent = None
        self.path = [self] # root 부터 여기까지 온 경로 ( 노드들 )
        self.root = self

        if nodes != None:
            self.Nodes += nodes
        
    def is_blossom(self):
        if len(self.Nodes) >= 3:
            return True
        return False
        
    def add_node(self, v):
        self.Nodes.append(v)
        
    def distance(self, v):
        return self.Graph.distance(self,v)
        
    def slack(self, v):
        return self.Graph.slack(self,v)
        
    def labeling(self): # label visited
        if self.parent == None:
            self.Label = True
        if self.Label != None:
            return
        self.Label = not self.parent.Label
    
    def is_containning(self, node):
        if self.Nodes == []: # base case
            if self == node:
                return True
        
        for x in self.Nodes:
            if x.is_containning(node):
                return True
        return False
    

In [ ]:
class blossomgraph:
    def __init__(self, indices):
        self.nodes = [ blossom( self, Id ) for Id in indices ]  # nodes 먼저 생성하고
        self.dist = {}
        self.tight_cache ={}
        
        for n1 in self.nodes:
            for n2 in self.nodes:
                if n1.Id >= n2.Id:
                    continue
                self.dist[(n1,n2)] = Decimal( format(distance_matrix[n1.Id][n2.Id], '.15f') )
        
    def distance(self, u, v):
        # 메모되어있음
        if u == v:
            return Decimal(0) 
        if self.dist.get((u,v)) != None:
            return self.dist[(u,v)]
        elif self.dist.get((v,u)) != None:
            return self.dist[(v,u)]
        tmp = Decimal('Infinity')
        for unode in u.Nodes:
            searchdist = self.distance(unode, v) - unode.Dual
            if searchdist < tmp:
                tmp = searchdist

        for vnode in v.Nodes:
            searchdist = self.distance(u, vnode) - vnode.Dual
            if searchdist < tmp:
                tmp = searchdist
                
        self.dist[(u,v)] = tmp
        return tmp
        
    def slack(self, u, v):
        return self.distance(u,v) - u.Dual - v.Dual
        

    def update_tightnode(self):
        self.tight_cache.clear()
        for node in self.nodes:
            tight_list = []
            for other in self.nodes:
                if node != other and self.slack(node, other) <= floaterr and self.slack(node, other) >= -floaterr:
                    tight_list.append(other)
            self.tight_cache[node.Id] = tight_list
        

    def tightnode(self,u):
        return self.tight_cache.get(u.Id, [])

    def init_nodes(self):
        for node in self.nodes:
            node.Visited = False
            node.Label = None
            node.parent = None
    def connect(self,u,v):
        u.Matching = v
        v.Matching = u

    def find_cycle(self, u,v):
        upath =[]
        vpath = []
        upath.append(u)
        vpath.append(v)

        cur = u
        while cur.parent != None:
            upath.append( cur.parent )
            cur = cur.parent
        cur = v
        while cur.parent != None:
            vpath.append( cur.parent )
            cur = cur.parent
        cycle = []
        for uindex in range(len(upath)):
            if upath[uindex] in vpath:
                break
        target = upath[uindex]
        cur = u
        while cur != target:
            cycle.append( cur )
            cur = cur.parent
        cycle.append(target)
    
        cur = v
        while cur != target:
            cycle.insert(0, cur )
            cur = cur.parent
        while cycle[0] != target:
            cycle.append( cycle.pop(0) )
        return cycle

    def SHRINK(self, cycle):
        global next_id
        newblossom = blossom(self, next_id)
        next_id += 1
        newblossom.Nodes = cycle 
        self.nodes.append(newblossom) 
        for node in cycle:
            self.nodes.remove(node)
        return newblossom

    def EXPAND(self, blossom):
        closenode = None
        tmp = Decimal('Infinity')
        for node in blossom.Nodes:
            dist = self.distance(node, blossom.Matching)
            if dist < tmp:
                tmp = dist
                closenode = node
        while blossom.Nodes[0] != closenode:
            blossom.Nodes.append( blossom.Nodes.pop(0) )
        blossom.Nodes[0].Matching = blossom.Matching
        blossom.Matching.Matching = blossom.Nodes[0]
        for idx in range(1, len(blossom.Nodes) -1 , 2):
            self.connect( blossom.Nodes[idx], blossom.Nodes[idx+1] )
        self.nodes.remove(blossom)

        for node in blossom.Nodes:
            self.nodes.append(node)

    def bfs(self, root):
        for node in self.nodes:
            node.Visited = False

        queue = []
        root.Label = True
        root.root = root
        root.path = [root]

        queue.append(root)
        root.Visited = True
        
        print("root id : ", root.Id)

        while queue:
            curnode = queue.pop(0)
            
            print("curnode : ",curnode.Id)
            print("tight nodes : ", [ x.Id for x in self.tightnode(curnode)] )
            for node in [node for node in self.tightnode(curnode) if node.Matching != curnode]:
                if curnode.Label == True and node.Label == True and node.root == curnode.root: # shrink
                    print("shrink")
                    
                    cycle = self.find_cycle(curnode, node)

                    newblossom = self.SHRINK(cycle)
                    
                    newblossom.root = curnode.root
                    newblossom.path = curnode.path + [newblossom]
                    newblossom.Visited = True
                    newblossom.labeling()                    
                    newblossom.parent = cycle[0].parent
                    
                    if newblossom.parent != None:
                        newblossom.Matching = newblossom.parent
                        newblossom.Matching.Matching = newblossom
                    else:
                        newblossom.Matching = None


                    if root in cycle:
                        newblossom.root = newblossom
                        newblossom.path = [newblossom]

                    self.bfsall()
                    return

                elif node.Label == True and curnode.Label == True and node.root != curnode.root: # augment
                    print("augment" , node.Id)

                    augpath = curnode.path
                    for x in range(len(node.path)-1,-1,-1):
                        augpath.append( node.path[x] )
                    print("augpath", [x.Id for x in augpath])

                    self.free(curnode.root)
                    self.free(node.root)

                    for idx in range(0, len(augpath),2):
                        u = augpath[idx]
                        v = augpath[idx+1]
                        self.connect(u,v)

                        u.Label = None
                        u.path = []
                        u.parent = None
                        u.Visited = False
                        u.root = None

                        v.Label = None
                        v.path = []
                        v.parent = None
                        v.Visited = False
                        v.root = None

                    return 1
                    


                elif ( node.Label == None and node.Matching != None ): # grow
                    print("grow" , node.Id)
                    
                    node.parent = curnode
                    node.Matching.parent = node
    
                    node.labeling()
                    node.Matching.labeling()

                    node.Visited = True
                    node.Matching.Visited = True
                    queue.append(node.Matching)

                    node.root = curnode.root
                    node.Matching.root = curnode.root

                    node.path = curnode.path + [node]
                    node.Matching.path = node.path + [node.Matching]

                    if node.Label == False and node.is_blossom() and node.Dual == 0:
                        self.EXPAND(node)
                        self.bfsall()
                        return
                else:
                    print("bfs err")
        return 0
    
    def bfsall(self):
        self.update_tightnode()
        for node in self.nodes:
                node.parent = None
                node.Label = None
                node.Visited = None
                node.root = None
        freenode = [x for x in self.nodes if x.Matching == None]
        for node in freenode:
            if node.Matching != None:
                    continue
            self.bfs(node)

    def updatedual(self):
        delta = Decimal('Infinity')
        pnodes = [ node for node in self.nodes if node.Label == True ]
        blossom = [ node for node in self.nodes if node.is_blossom() and node.Label == False]
        freenodes = [ node for node in self.nodes if node.Label == None ]
        case = None
        tmp = None
        nextroot = None
        for n1 in self.nodes:
            for n2 in self.nodes:
                if n1.Id >= n2.Id:
                    continue
                elif (n1.Label == True and n2.Label == None) or (n1.Label == None and n2.Label == True):
                    if delta > self.slack(n1, n2):
                        delta = self.slack(n1, n2)
                        case = "grow"
                        tmp = (n1.Id, n2.Id)
                        nextroot = n1.root if n1.Label == True else n2.root
                elif (n1.root != n2.root) and (n1.Label == True and n2.Label == True):
                    if delta > self.slack(n1, n2) / 2:
                        delta = self.slack(n1, n2)/2
                        case = "augment"
                        tmp = (n1.Id, n2.Id)
                        nextroot = n1.root
                elif (n1.Label == True and n2.Label == True) and (n1.root == n2.root):
                    if delta > self.slack(n1, n2) / 2:
                        delta = self.slack(n1, n2) / 2
                        case = "shrink"
                        tmp = (n1.Id, n2.Id)
                        nextroot = n1.root

        for b1 in blossom:
            if b1.Label == True:
                continue
            if b1.Label == False and delta > b1.Dual:
                    case = "Expand"
                    delta = b1.Dual
                    tmp = b1.Id
                    nextroot = b1.root
    

        print( case, tmp )
        for node in self.nodes:
            if node.Label == True:
                node.Dual += delta
            elif node.Label == False:
                node.Dual -= delta

        return nextroot

    def free(self, root):
        for node in self.nodes:
            if node.root == root:
                node.Label = None
                node.path = []
                node.parent = None
                node.Visited = False
            
    def find_min_weight_matching(self):
        for node in self.nodes:
            m = Decimal('Infinity')
            for To in self.nodes:
                if To == node:
                    continue
                if m > node.distance(To):
                    m = node.distance(To)
            node.Dual = m / 2
        
        for n1 in self.nodes:
            for n2 in self.nodes:
                if self.slack(n1, n2) < floaterr and self.slack(n1, n2) > -floaterr and n1.Matching == None and n2.Matching == None:
                    self.connect(n1, n2)
                    n1.Label = None
                    n1.path = []
                    n1.parent = None
                    n1.Visited = False
                    n2.Label = None
                    n2.path = []
                    n2.parent = None
                    n2.Visited = False
                    n1.root = None
                    n2.root = None
        return


f = blossomgraph(odd_rows)
f.find_min_weight_matching()


In [ ]:
# def get_node_pos(node, points, pos_cache):
#     # 이미 계산된 경우 캐시 사용
#     if node.Id in pos_cache:
#         return pos_cache[node.Id]
#     if not node.is_blossom():
#         pos_cache[node.Id] = tuple(points[node.Id])
#         return pos_cache[node.Id]
#     # blossom 노드라면 내부 노드들의 좌표 평균
#     xs, ys = [], []
#     for n in node.Nodes:
#         x, y = get_node_pos(n, points, pos_cache)
#         xs.append(x)
#         ys.append(y)
#     pos_cache[node.Id] = (np.mean(xs), np.mean(ys))
#     return pos_cache[node.Id]

def get_node_pos(node, points, pos_cache):
    """
    node      : blossomgraph의 노드 객체
    points    : 원본 점들의 (N,2) NumPy 배열
    pos_cache : 계산된 위치를 저장하는 dict
    """
    # 1) 캐시에 있으면 바로 반환
    if node.Id in pos_cache:
        return pos_cache[node.Id]
    
    # 2) 원본 노드인 경우 (ID가 points 범위 안에 있을 때)
    if node.Id < len(points):
        pos_cache[node.Id] = tuple(points[node.Id])
        return pos_cache[node.Id]
    
    # 3) 블라썸 노드인 경우: 내부 노드들의 좌표 평균
    xs, ys = [], []
    for n in node.Nodes:
        x, y = get_node_pos(n, points, pos_cache)
        xs.append(x)
        ys.append(y)
    pos_cache[node.Id] = (np.mean(xs), np.mean(ys))
    return pos_cache[node.Id]

# def visualize(f, points):
#     # 노드별 위치 계산
#     pos_cache = {}
#     for node in f.nodes:
#         get_node_pos(node, points, pos_cache)

#     # 나머지 시각화 로직은 그대로...
#     plt.figure(figsize=(10, 10))
#     # 매칭 간선, 타이트 간선, 노드 원 그리기 등


def visualize(f, points):
    """
    f      : blossomgraph 인스턴스
    points : 각 원래 노드(Id)에 대응하는 (x, y) 좌표를 담은 dict 또는 배열
    """
    # 1) 노드별 위치 계산(pos): 블라썸이면 포함된 노드들의 평균 위치
    pos = {}
    for node in f.nodes:
        get_node_pos(node, points, pos)
        
    # pos = {}
    # for node in f.nodes:
    #     if node.is_blossom():
    #         xs = [points[n.Id][0] for n in node.Nodes]
    #         ys = [points[n.Id][1] for n in node.Nodes]
    #         pos[node.Id] = (np.mean(xs), np.mean(ys))
    #     else:
    #         pos[node.Id] = tuple(points[node.Id])

    plt.figure(figsize=(10, 10))

    # 2) 매칭 간선 먼저 그리기 (두꺼운 검은 선)
    drawn = set()
    for node in f.nodes:
        match = node.Matching
        if match is not None and (node.Id, match.Id) not in drawn:
            x1, y1 = pos[node.Id]
            x2, y2 = pos[match.Id]
            plt.plot([x1, x2], [y1, y2],
                     color='k', linewidth=4)
            drawn.add((node.Id, match.Id))
            drawn.add((match.Id, node.Id))
            
    for node in f.nodes:
        for tnode in f.tightnode(node):
            if node.Id < tnode.Id:  # 중복 방지
                x1, y1 = pos[node.Id]
                x2, y2 = pos[tnode.Id]
                plt.plot([x1, x2], [y1, y2], color='b', linestyle='dashed', linewidth=1, alpha=0.5)
    # 3) 노드(원 + 점 + 아이디) 그리기
    ax = plt.gca()
    for node in f.nodes:
        x, y = pos[node.Id]
        radius = node.Dual
        circle = plt.Circle((x, y), radius,
                            color='b', fill=False, alpha=0.5)
        ax.add_patch(circle)
        plt.plot(x, y, 'ro')
        plt.text(x, y, str(node.Id),
                 fontsize=12, ha='right', va='bottom',
                 color='black')

    plt.title('Graph nodes with dual value as radius and matching edges')
    plt.xlabel('X'); plt.ylabel('Y')
    plt.axis('equal'); plt.grid(True)
    plt.show()

In [ ]:
def printnodes():
    for x in f.nodes:
        if x.root == None:
            if x.Matching == None:
                print("Id : ",x.Id, x.Matching,"\n    Root : ", x.root,"\n    Label : ", x.Label, "\n    path: ", [ y.Id for y in x.path])
            else:
                print("Id : ",x.Id, x.Matching.Id,"\n    Root : ", x.root,"\n    Label : ", x.Label, "\n    path: ", [ y.Id for y in x.path])
        else:
            if x.Matching == None:
                print("Id : ",x.Id, x.Matching,"\n    Root : ", x.root.Id,"\n    Label : ", x.Label, "\n    path: ", [ y.Id for y in x.path])
            else:
                print("Id : ",x.Id, x.Matching.Id,"\n    Root : ", x.root.Id,"\n    Label : ", x.Label, "\n    path: ", [ y.Id for y in x.path])

printnodes()

import time


In [ ]:
start = time.time()

In [ ]:
f.bfsall()

In [ ]:
while 1:
    f.bfsall()
    cond = f.updatedual()
    # visualize(f,points)
    if cond == None:
        break

In [ ]:
blossom = []
for node in f.nodes:
    if node.is_blossom():
        blossom.append(node)
blossom.sort(key=lambda x: x.Id)
if blossom != []:
    blossom = blossom[0] # 원래 -1
    f.EXPAND(blossom)
    # visualize(f,points)
while blossom != []:
    blossom = []
    for node in f.nodes:
        if node.is_blossom():
            blossom.append(node)
    blossom.sort(key=lambda x: x.Id)
    if blossom != []:
        blossom = blossom[-1]
        f.EXPAND(blossom)
        # visualize(f,points)

In [ ]:
f.update_tightnode()
visualize(f,points)

In [ ]:
len( odd_rows)

In [ ]:
sum =0
for x in f.nodes:
    if x.Matching != None:
        sum += distance_matrix[x.Id][x.Matching.Id] # f.distance(x, x.Matching)
    else:
        print("err")


In [ ]:
print("sum : ", sum/2)

In [ ]:
matching = []
for x in f.nodes:
    if x.Matching.Id > x.Id:
        matching.append( (x.Id, x.Matching.Id) )

matching.sort( key=lambda x: x[0] )
matching

In [ ]:
mark = {}
for x in odd_rows:
    mark[x] = False

for node in f.nodes:
    mark[node.Id] = True
    mark[node.Matching.Id] = True

for node in f.nodes:
    if node != node.Matching.Matching:
        print("err")


In [ ]:
nodes = []
edges=[]
for x in range( len(mstgraph) ):
    for y in range( len(mstgraph) ):
        if x >= y:
            continue
        if mstgraph[x][y] != np.inf:
            edges.append( (x,y) )
for x in range(len(mstgraph)):
    nodes.append(x)
    
edges = edges+matching
print(edges)
print(nodes)

In [ ]:
def find_eulerian_circuit(nodes, edges):
    stack = [nodes[0]]
    result =[]
    while stack:
        node = stack[-1]
        found = False
        for edge in edges:
            if edge[0] == node:
                stack.append(edge[1])
                edges.remove(edge)
                found = True
                break
            elif edge[1] == node:
                stack.append(edge[0])
                edges.remove(edge)
                found = True
                break
        if not found:
            result.append(stack.pop())
    return result

In [ ]:
e_path = find_eulerian_circuit(nodes, edges)

In [ ]:
short_cutting = []
for i in range(len(e_path) - 1):
    if e_path[i] not in short_cutting:
        short_cutting.append(e_path[i])

In [ ]:
dist = 0
for i in range(len(short_cutting) -1):
    dist += distance_matrix[short_cutting[i]][short_cutting[i+1]]

In [ ]:
dist

end = time.time()
print("Execution Time: ", end - start)
